In [1]:
import numpy as np

class Value:
    def __init__(self,val,parents=None,op='',children=None):
        if parents is None:
            parents=[]
        if children is None:
            children=[]
        self.value=val
        self.parents=parents
        self.children=children
        self.op=op
        self.grad=0
        self._gradient=lambda:None
    def __repr__(self):
        return f'Value of {self.value}'
    def __add__(self,other):
        result=Value(self.value+other.value,[self,other],'+')
        self.children.append(result)
        other.children.append(result)
        def _gradient():
            self.grad+=result.grad
            other.grad+=result.grad
        result._gradient=_gradient
        return result
    def __mul__(self,other):
        result=Value(self.value*other.value,[self,other],'*')
        self.children.append(result)
        other.children.append(result)
        def _gradient():
            self.grad+=other.value*result.grad
            other.grad+=self.value*result.grad
        result._gradient=_gradient
        return result
    def __truediv__(self,other):
        if other.value==0:
            raise ZeroDivisionError
        result=Value(self.value/other.value,[self,other],'/')
        self.children.append(result)
        other.children.append(result)
        def _gradient():
            self.grad+=result.grad/other.value
            other.grad-=self.value*result.grad/other.value**2
        result._gradient=_gradient
        return result
    def exp(self):
        result=Value(np.exp(self.value),[self],'exp')
        self.children.append(result)
        def _gradient():
            self.grad+=result.value*result.grad
        result._gradient=_gradient
        return result
    def log(self):
        result=Value(np.log(self.value),[self],'log')
        self.children.append(result)
        def _gradient():
            self.grad+=result.grad/self.value
        result._gradient=_gradient
        return result
    def sigmoid(self):
        negation=Value(-1)
        intermediate1=negation*self
        intermediate2=intermediate1.exp()
        numerator=Value(1)
        denominator=Value(1)+intermediate2
        result=numerator/denominator
        return result
    def gradient(self):
        topo=[]
        visited=set()
        def build(value):
            if value not in visited:
                visited.add(value)
                for parent in value.parents:
                    build(parent)
                topo.append(value)
        build(self)
        self.grad=1
        for value in reversed(topo):
            value._gradient()
    def remove_gradient(self):
        self.grad=0
        for p in self.parents:
            p.remove_gradient()

In [2]:
import random

class Matrix:
    def __init__(self,rows,cols):
        self.rows=rows
        self.cols=cols
        self.matrix=[[Value(random.uniform(-1,1)) for col in range(cols)] for row in range(rows)]
    def __repr__(self):
        return str(self.matrix)
    def __add__(self,other):
        if self.rows==other.rows and self.cols==other.cols:
            result=Matrix(self.rows,self.cols)
            for row in range(self.rows):
                for col in range(self.cols):
                    result.matrix[row][col]=self.matrix[row][col]+other.matrix[row][col]
            return result
        elif self.rows==other.rows and other.cols==1:
            result=Matrix(self.rows,self.cols)
            for row in range(self.rows):
                for col in range(self.cols):
                    result.matrix[row][col]=self.matrix[row][col]+other.matrix[row][0]
            return result
        elif self.cols==other.cols and other.rows==1:
            result=Matrix(self.rows,self.cols)
            for row in range(self.rows):
                for col in range(self.cols):
                    result.matrix[row][col]=self.matrix[row][col]+other.matrix[0][col]
            return result
        else:
            raise ValueError
    def __mul__(self,other):
        if self.rows==other.rows and self.cols==other.cols:
            result=Matrix(self.rows,self.cols)
            for row in range(self.rows):
                for col in range(self.cols):
                    result.matrix[row][col]=self.matrix[row][col]*other.matrix[row][col]
            return result
        elif self.rows==other.rows and other.cols==1:
            result=Matrix(self.rows,self.cols)
            for row in range(self.rows):
                for col in range(self.cols):
                    result.matrix[row][col]=self.matrix[row][col]*other.matrix[row][0]
            return result
        elif self.cols==other.cols and other.rows==1:
            result=Matrix(self.rows,self.cols)
            for row in range(self.rows):
                for col in range(self.cols):
                    result.matrix[row][col]=self.matrix[row][col]*other.matrix[0][col]
            return result
        else:
            raise ValueError
    def __matmul__(self,other):
        if self.cols==other.rows:
            result=Matrix(self.rows,other.cols)
            for row in range(self.rows):
                for col in range(other.cols):
                    result.matrix[row][col].value=0
                    for i in range(self.cols):
                        result.matrix[row][col]+=self.matrix[row][i]*other.matrix[i][col]
            return result
        else:
            raise ValueError
    def __truediv__(self,other):
        if self.rows==other.rows and self.cols==other.cols:
            result=Matrix(self.rows,self.cols)
            for row in range(self.rows):
                for col in range(self.cols):
                    result.matrix[row][col]=self.matrix[row][col]/other.matrix[row][col]
            return result
        elif self.rows==other.rows and other.cols==1:
            result=Matrix(self.rows,self.cols)
            for row in range(self.rows):
                for col in range(self.cols):
                    result.matrix[row][col]=self.matrix[row][col]/other.matrix[row][0]
            return result
        elif self.cols==other.cols and other.rows==1:
            result=Matrix(self.rows,self.cols)
            for row in range(self.rows):
                for col in range(self.cols):
                    result.matrix[row][col]=self.matrix[row][col]/other.matrix[0][col]
            return result
        elif other.rows==1 and other.cols==1:
            result=Matrix(self.rows,self.cols)
            for row in range(self.rows):
                for col in range(self.cols):
                    result.matrix[row][col]=self.matrix[row][col]/other.matrix[0][0]
            return result
        else:
            raise ValueError
    def exp(self):
        for row in range(self.rows):
            for col in range(self.cols):
                self.matrix[row][col]=self.matrix[row][col].exp()
        return self
    def sigmoid(self):
        for row in range(self.rows):
            for col in range(self.cols):
                self.matrix[row][col]=self.matrix[row][col].sigmoid()
        return self
    def negative_log(self):
        for row in range(self.rows):
            for col in range(self.cols):
                self.matrix[row][col]=Value(-1)*self.matrix[row][col].log()
        return self
    def learn(self,learning_rate):
        for row in range(self.rows):
            for col in range(self.cols):
                self.matrix[row][col].value-=learning_rate*self.matrix[row][col].grad

In [3]:
import csv

x=[]
y=[]
with open('iris.csv') as f:
    reader=csv.reader(f)
    next(reader)
    for row in reader:
        features=list(map(float,row[1:5]))
        label=row[5]
        x.append(features)
        y.append(label)
label_map={'Iris-setosa':0,'Iris-versicolor':1,'Iris-virginica':2}
y=[label_map[v] for v in y]
print(x[:4])
print(y[:4])

[[5.1, 3.5, 1.4, 0.2], [4.9, 3.0, 1.4, 0.2], [4.7, 3.2, 1.3, 0.2], [4.6, 3.1, 1.5, 0.2]]
[0, 0, 0, 0]


In [4]:
x_train=x[0:40]+x[50:90]+x[100:140]
y_train=y[0:40]+y[50:90]+y[100:140]
x_test=x[40:50]+x[90:100]+x[140:150]
y_test=y[40:50]+y[90:100]+y[140:150]

In [5]:
means=[0,0,0,0]
for row in range(120):
    for col in range(4):
        means[col]+=x_train[row][col]
means=[mean/120 for mean in means]
stds=[0,0,0,0]
for row in range(120):
    for col in range(4):
        stds[col]+=(x_train[row][col]-means[col])**2
stds=[(std/120)**0.5 for std in stds]
for row in range(120):
    for col in range(4):
        x_train[row][col]=(x_train[row][col]-means[col])/stds[col]
for row in range(30):
    for col in range(4):
        x_test[row][col]=(x_test[row][col]-means[col])/stds[col]

In [6]:
inputs1=Matrix(120,4)
for row in range(120):
    for col in range(4):
        inputs1.matrix[row][col].value=x_train[row][col]
weights1=Matrix(4,10)
biases1=Matrix(1,10)
weights2=Matrix(10,10)
biases2=Matrix(1,10)
weights3=Matrix(10,3)
biases3=Matrix(1,3)
epsilon=Matrix(120,3)
large_num=Matrix(1,1)
large_num.matrix[0][0].value=1e9
epsilon=epsilon/large_num
selection=Matrix(120,3)
for row in range(120):
    for col in range(3):
        if col==y_train[row]:
            selection.matrix[row][col]=Value(1)
        else:
            selection.matrix[row][col]=Value(0)

In [21]:
import csv

def store(parameters,filename):
    with open(filename,'w',newline='\n') as f:
        writer=csv.writer(f)
        for row in range(parameters.rows):
            values=[]
            for col in range(parameters.cols):
                values.append(parameters.matrix[row][col].value)
            writer.writerow(values)

safeguard=input('WARNING: OVERWRITING EXISTING FILES. TYPE yes TO CONTINUE:')
if safeguard=='yes':
    store(weights1,'weights1.csv')
    store(biases1,'biases1.csv')
    store(weights2,'weights2.csv')
    store(biases2,'biases2.csv')
    store(weights3,'weights3.csv')
    store(biases3,'biases3.csv')

In [38]:
learning_rate=0.1

for step in range(1000):
    inputs2=inputs1@weights1+biases1
    inputs2=inputs2.sigmoid()
    inputs3=inputs2@weights2+biases2
    inputs3=inputs3.sigmoid()
    outputs=inputs3@weights3+biases3
    for row in range(120):
        maximum=Value(-max(outputs.matrix[row][0].value,outputs.matrix[row][1].value,outputs.matrix[row][2].value))
        outputs.matrix[row][0]+=maximum
        outputs.matrix[row][1]+=maximum
        outputs.matrix[row][2]+=maximum
    outputs=outputs.exp()
    for row in range(120):
        norm=outputs.matrix[row][0]+outputs.matrix[row][1]+outputs.matrix[row][2]
        outputs.matrix[row][0]=outputs.matrix[row][0]/norm
        outputs.matrix[row][1]=outputs.matrix[row][1]/norm
        outputs.matrix[row][2]=outputs.matrix[row][2]/norm
    outputs=(outputs+epsilon).negative_log()
    outputs=outputs*selection
    loss=Value(0)
    for row in range(120):
        for col in range(3):
            loss+=outputs.matrix[row][col]
    loss=loss/Value(120)
    print(step,'\t',loss)
    loss.gradient()
    weights1.learn(learning_rate)
    biases1.learn(learning_rate)
    weights2.learn(learning_rate)
    biases2.learn(learning_rate)
    weights3.learn(learning_rate)
    biases3.learn(learning_rate)
    loss.remove_gradient()
    if step%20==0:
        store(weights1,'weights1.csv')
        store(biases1,'biases1.csv')
        store(weights2,'weights2.csv')
        store(biases2,'biases2.csv')
        store(weights3,'weights3.csv')
        store(biases3,'biases3.csv')

0 	 Value of 0.37191015818267836
1 	 Value of 0.3710367268117644
2 	 Value of 0.37016679821958454
3 	 Value of 0.369300336178575
4 	 Value of 0.3684373051595904
5 	 Value of 0.36757767031770644
6 	 Value of 0.3667213974782998
7 	 Value of 0.3658684531233963
8 	 Value of 0.3650188043782813
9 	 Value of 0.36417241899837394
10 	 Value of 0.36332926535635135
11 	 Value of 0.36248931242952726
12 	 Value of 0.36165252978747414
13 	 Value of 0.3608188875798885
14 	 Value of 0.3599883565246904
15 	 Value of 0.359160907896359
16 	 Value of 0.35833651351449447
17 	 Value of 0.35751514573260484
18 	 Value of 0.3566967774271132
19 	 Value of 0.3558813819865806
20 	 Value of 0.3550689333011419
21 	 Value of 0.35425940575215037
22 	 Value of 0.35345277420202464
23 	 Value of 0.3526490139842983
24 	 Value of 0.35184810089386803
25 	 Value of 0.3510500111774306
26 	 Value of 0.35025472152411413
27 	 Value of 0.3494622090562933
28 	 Value of 0.3486724513205893
29 	 Value of 0.34788542627904584
30 	 Val

KeyboardInterrupt: 

In [8]:
import csv

def retrieve(parameters,filename):
    with open(filename,'r') as f:
        reader=csv.reader(f)
        row=0
        for values in reader:
            for col in range(len(values)):
                parameters.matrix[row][col].value=float(values[col])
            row+=1

retrieve(weights1,'weights1.csv')
retrieve(biases1,'biases1.csv')
retrieve(weights2,'weights2.csv')
retrieve(biases2,'biases2.csv')
retrieve(weights3,'weights3.csv')
retrieve(biases3,'biases3.csv')

In [9]:
inputs1=Matrix(30,4)
for row in range(30):
    for col in range(4):
        inputs1.matrix[row][col].value=x_test[row][col]
inputs2=inputs1@weights1+biases1
inputs2=inputs2.sigmoid()
inputs3=inputs2@weights2+biases2
inputs3=inputs3.sigmoid()
outputs=inputs3@weights3+biases3
for row in range(30):
    maximum=Value(-max(outputs.matrix[row][0].value,outputs.matrix[row][1].value,outputs.matrix[row][2].value))
    outputs.matrix[row][0]+=maximum
    outputs.matrix[row][1]+=maximum
    outputs.matrix[row][2]+=maximum
outputs=outputs.exp()
for row in range(30):
    norm=outputs.matrix[row][0]+outputs.matrix[row][1]+outputs.matrix[row][2]
    outputs.matrix[row][0]=outputs.matrix[row][0]/norm
    outputs.matrix[row][1]=outputs.matrix[row][1]/norm
    outputs.matrix[row][2]=outputs.matrix[row][2]/norm
for row in range(30):
    print(y_test[row],end='\t')
    if outputs.matrix[row][0].value>outputs.matrix[row][1].value and outputs.matrix[row][0].value>outputs.matrix[row][2].value:
        print('setosa',end='\t\t')
    elif outputs.matrix[row][1].value>outputs.matrix[row][0].value and outputs.matrix[row][1].value>outputs.matrix[row][2].value:
        print('versicolor',end='\t')
    else:
        print('virginica',end='\t')
    print(outputs.matrix[row][0].value,'\t',outputs.matrix[row][1].value,'\t',outputs.matrix[row][2].value)

0	setosa		0.9165352237090076 	 0.07971877764759415 	 0.0037459986433981307
0	setosa		0.7555246553096961 	 0.23574134597028534 	 0.0087339987200185
0	setosa		0.9011336447382094 	 0.09469675430461641 	 0.0041696009571741905
0	setosa		0.8979832666073785 	 0.09704690022844487 	 0.004969833164176752
0	setosa		0.9093672637669004 	 0.08608598872837783 	 0.004546747504721701
0	setosa		0.884678849612485 	 0.11055508390721061 	 0.004766066480304301
0	setosa		0.9210400180729172 	 0.07526161110197922 	 0.00369837082510357
0	setosa		0.9017806790193661 	 0.09406001679238915 	 0.0041593041882446264
0	setosa		0.92122415481979 	 0.0751301814524713 	 0.003645663727738724
0	setosa		0.9113469685578759 	 0.08481276447313495 	 0.003840266968989206
1	versicolor	0.07588403340568005 	 0.7112176214531797 	 0.2128983451411402
1	versicolor	0.05781361350673068 	 0.5938810411160704 	 0.348305345377199
1	versicolor	0.10898822903748298 	 0.7193112711697707 	 0.1717004997927463
1	versicolor	0.24660970890574915 	 0.687